In [0]:
df = spark.read.csv(
"/Volumes/sales_catalog/sales_schema/sales_volume/sales_data.csv",
header=True,
inferSchema=True,
sep=";"
)

display(df)

order_id,date,customer_id,product,category,price,quantity,total_amount,city
1,2024-08-31,1082,Desk,Furniture,1200,4,4800,Belo Horizonte
2,2024-03-23,1065,Desk,Furniture,1200,3,3600,Curitiba
3,2024-02-19,1132,Mouse,Electronics,80,3,240,Porto Alegre
4,2024-12-30,1163,Mouse,Electronics,80,1,80,São Paulo
5,2024-10-25,1083,Desk,Furniture,1200,3,3600,São Paulo
6,2024-04-22,1122,Monitor,Electronics,900,3,2700,Porto Alegre
7,2024-11-13,1035,Monitor,Electronics,900,4,3600,Curitiba
8,2024-11-25,1049,Laptop,Electronics,3500,3,10500,São Paulo
9,2024-01-16,1003,Backpack,Accessories,180,1,180,Rio de Janeiro
10,2024-11-09,1108,Keyboard,Electronics,150,3,450,Rio de Janeiro


Databricks visualization. Run in Databricks to view.

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("sales_bronze")

In [0]:
display(spark.read.table("sales_bronze"))

order_id,date,customer_id,product,category,price,quantity,total_amount,city
1,2024-08-31,1082,Desk,Furniture,1200,4,4800,Belo Horizonte
2,2024-03-23,1065,Desk,Furniture,1200,3,3600,Curitiba
3,2024-02-19,1132,Mouse,Electronics,80,3,240,Porto Alegre
4,2024-12-30,1163,Mouse,Electronics,80,1,80,São Paulo
5,2024-10-25,1083,Desk,Furniture,1200,3,3600,São Paulo
6,2024-04-22,1122,Monitor,Electronics,900,3,2700,Porto Alegre
7,2024-11-13,1035,Monitor,Electronics,900,4,3600,Curitiba
8,2024-11-25,1049,Laptop,Electronics,3500,3,10500,São Paulo
9,2024-01-16,1003,Backpack,Accessories,180,1,180,Rio de Janeiro
10,2024-11-09,1108,Keyboard,Electronics,150,3,450,Rio de Janeiro


In [0]:
from pyspark.sql.functions import col

df = spark.read.table("sales_bronze")

df_clean = (
    df
    .dropna()
    .withColumn("price", col("price").cast("double"))
    .withColumn("quantity", col("quantity").cast("int"))
)

df_clean.write.format("delta").mode("overwrite").saveAsTable("sales_silver")

In [0]:
%sql
CREATE TABLE sales_gold_category AS
SELECT
category,
SUM(total_amount) AS revenue
FROM sales_silver
GROUP BY category

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8011948207113276>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE TABLE sales_gold_category AS\nSELECT\ncategory,\nSUM(total_amount) AS revenue\nFROM sales_silver\nGROUP BY category\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:192, in SqlMagic.sql(self, li

In [0]:
%sql
CREATE TABLE sales_gold_month AS
SELECT
date_trunc('month', date) AS month,
SUM(total_amount) AS revenue
FROM sales_silver
GROUP BY month

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8011948207113277>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE TABLE sales_gold_month AS\nSELECT\ndate_trunc('month', date) AS month,\nSUM(total_amount) AS revenue\nFROM sales_silver\nGROUP BY month\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:192, in S

In [0]:
%sql
CREATE TABLE sales_gold_products AS
SELECT
product,
SUM(total_amount) AS revenue
FROM sales_silver
GROUP BY product
ORDER BY revenue DESC

In [0]:
%python

df = spark.sql("""
SELECT
category,
SUM(total_amount) AS revenue
FROM sales_silver
GROUP BY category
""")

display(df)

In [0]:
df = spark.sql("""
SELECT
category,
SUM(total_amount) AS revenue
FROM sales_silver
GROUP BY category
""")

display(df)

In [0]:
%sql
SELECT * 
FROM sales_silver
LIMIT 10

In [0]:
sales_by_category = df.groupBy("category").count()
display(sales_by_category)

In [0]:
sales_by_date = df.groupBy("date").count()
display(sales_by_date)

In [0]:
top_products = df.groupBy("product").count().orderBy("count", ascending=False)
display(top_products)

In [0]:
orders_by_customer = df.groupBy("customer_id").count().orderBy("count", ascending=False)
display(orders_by_customer)

In [0]:
df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/sales_catalog/sales_schema/sales_volume/sales_data.csv")

display(df)




In [0]:
# vendas por produto
display(df.groupBy("product").count())

# vendas por cliente
display(df.groupBy("customer_id").count())

# vendas por data
display(df.groupBy("date").count())

# vendas por categoria
display(df.groupBy("category").count())

# produtos mais vendidos
display(df.groupBy("product").sum("sales").orderBy("sum(sales)", ascending=False))


In [0]:
df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.option("delimiter",";") \
.load("/Volumes/sales_catalog/sales_schema/sales_volume/sales_data.csv")

display(df)



In [0]:
df.printSchema()


In [0]:
display(df)


In [0]:
# ==============================
# 1. IMPORTAR FUNÇÕES
# ==============================

from pyspark.sql.functions import sum, month

# ==============================
# 2. VERIFICAR DADOS
# ==============================

display(df)
df.printSchema()

# ==============================
# 3. KPI - RECEITA TOTAL
# ==============================

total_revenue = df.selectExpr("sum(total_amount) as total_revenue")
display(total_revenue)

# ==============================
# 4. RECEITA POR CATEGORIA
# ==============================

revenue_by_category = df.groupBy("category") \
    .sum("total_amount") \
    .orderBy("sum(total_amount)", ascending=False)

display(revenue_by_category)

# ==============================
# 5. TOP PRODUTOS MAIS VENDIDOS
# ==============================

top_products = df.groupBy("product") \
    .sum("quantity") \
    .orderBy("sum(quantity)", ascending=False)

display(top_products)

# ==============================
# 6. RECEITA POR CIDADE
# ==============================

revenue_by_city = df.groupBy("city") \
    .sum("total_amount") \
    .orderBy("sum(total_amount)", ascending=False)

display(revenue_by_city)

# ==============================
# 7. RECEITA POR DATA
# ==============================

revenue_by_date = df.groupBy("date") \
    .sum("total_amount") \
    .orderBy("date")

display(revenue_by_date)

# ==============================
# 8. RECEITA POR MÊS
# ==============================

revenue_by_month = df.groupBy(month("date").alias("month")) \
    .sum("total_amount") \
    .orderBy("month")

display(revenue_by_month)

# ==============================
# 9. QUANTIDADE VENDIDA POR CATEGORIA
# ==============================

quantity_by_category = df.groupBy("category") \
    .sum("quantity") \
    .orderBy("sum(quantity)", ascending=False)

display(quantity_by_category)


order_id,date,customer_id,product,category,price,quantity,total_amount,city
1,2024-08-31,1082,Desk,Furniture,1200,4,4800,Belo Horizonte
2,2024-03-23,1065,Desk,Furniture,1200,3,3600,Curitiba
3,2024-02-19,1132,Mouse,Electronics,80,3,240,Porto Alegre
4,2024-12-30,1163,Mouse,Electronics,80,1,80,São Paulo
5,2024-10-25,1083,Desk,Furniture,1200,3,3600,São Paulo
6,2024-04-22,1122,Monitor,Electronics,900,3,2700,Porto Alegre
7,2024-11-13,1035,Monitor,Electronics,900,4,3600,Curitiba
8,2024-11-25,1049,Laptop,Electronics,3500,3,10500,São Paulo
9,2024-01-16,1003,Backpack,Accessories,180,1,180,Rio de Janeiro
10,2024-11-09,1108,Keyboard,Electronics,150,3,450,Rio de Janeiro


root
 |-- order_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: integer (nullable = true)
 |-- city: string (nullable = true)



total_revenue
439185


Databricks visualization. Run in Databricks to view.

category,sum(total_amount)
Electronics,319870
Furniture,107000
Accessories,10980
Stationery,1335


...

Databricks visualization. Run in Databricks to view.

month,sum(total_amount)
1,23650
2,48370
3,19370
4,61340
5,19145
6,59425
7,46065
8,15560
9,23685
10,45955


Databricks visualization. Run in Databricks to view.

category,sum(quantity)
Electronics,309
Furniture,115
Stationery,111
Accessories,61


Databricks visualization. Run in Databricks to view.

In [0]:
df.describe().show()


In [0]:
from pyspark.ml.regression import LinearRegression


In [0]:
# ==============================
# 1. IMPORTAR BIBLIOTECAS
# ==============================

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# ==============================
# 2. SELECIONAR COLUNAS NUMÉRICAS
# ==============================

data = df.select("price", "quantity", "total_amount")

# remover valores nulos para evitar erro
data = data.dropna()

display(data)

# ==============================
# 3. CRIAR COLUNA DE FEATURES
# ==============================

assembler = VectorAssembler(
    inputCols=["price", "quantity"],
    outputCol="features"
)

data_model = assembler.transform(data)

display(data_model)

# ==============================
# 4. CRIAR MODELO DE REGRESSÃO
# ==============================

lr = LinearRegression(
    featuresCol="features",
    labelCol="total_amount"
)

# ==============================
# 5. TREINAR MODELO
# ==============================

model = lr.fit(data_model)

# ==============================
# 6. FAZER PREVISÕES
# ==============================

predictions = model.transform(data_model)

display(predictions.select("price","quantity","total_amount","prediction"))
